# Credit Risk Scorecard — Bước 1: EDA & Data Cleaning
**Dataset:** Give Me Some Credit (Kaggle)  
**Mục tiêu:** Hiểu cấu trúc dữ liệu, xử lý missing values, outliers, và chuẩn bị dữ liệu sạch cho các bước tiếp theo.

---
### Cấu trúc notebook
1. Cài đặt thư viện & cấu hình
2. Load & kiểm tra dữ liệu thô
3. Phân tích biến mục tiêu (target)
4. Phân tích missing values
5. Phân tích outliers
6. Xử lý missing values
7. Xử lý outliers (Winsorization)
8. Kiểm tra sau xử lý & lưu file


## 1. Cài đặt thư viện & cấu hình

In [ ]:
# Cài đặt thư viện (chạy 1 lần, sau đó có thể comment lại)
# !pip install pandas numpy matplotlib seaborn scipy scikit-learn

import warnings
warnings.filterwarnings("ignore")

# ── Core libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats.mstats import winsorize
from sklearn.preprocessing import LabelEncoder

# ── Display settings ────────────────────────────────────────────────────────
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_rows", 60)

# ── Plot style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "figure.figsize": (12, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.titlepad": 12,
    "font.family": "sans-serif",
})
PALETTE = {"good": "#2196F3", "bad": "#E53935", "neutral": "#78909C"}

print("✓ Thư viện đã sẵn sàng")


## 2. Load & kiểm tra dữ liệu thô

> **Cách download dataset:**  
> 1. Vào https://www.kaggle.com/competitions/GiveMeSomeCredit/data  
> 2. Download file `cs-training.csv`  
> 3. Đặt vào cùng thư mục với notebook này


In [ ]:
# ── Load dữ liệu ────────────────────────────────────────────────────────────
DATA_PATH = "cs-training.csv"   # Thay đường dẫn nếu file ở thư mục khác

df_raw = pd.read_csv(DATA_PATH, index_col=0)

# ── Đổi tên cột cho dễ đọc ──────────────────────────────────────────────────
RENAME_MAP = {
    "SeriousDlqin2yrs":                        "target",
    "RevolvingUtilizationOfUnsecuredLines":     "revolving_util",
    "age":                                      "age",
    "NumberOfTime30-59DaysPastDueNotWorse":     "past_due_30_59",
    "DebtRatio":                                "debt_ratio",
    "MonthlyIncome":                            "monthly_income",
    "NumberOfOpenCreditLinesAndLoans":          "open_credit_lines",
    "NumberOfTimes90DaysLate":                  "late_90_days",
    "NumberRealEstateLoansOrLines":             "real_estate_loans",
    "NumberOfTime60-89DaysPastDueNotWorse":     "past_due_60_89",
    "NumberOfDependents":                       "dependents",
}
df_raw = df_raw.rename(columns=RENAME_MAP)

print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print("─" * 50)
df_raw.head()


In [ ]:
# ── Tổng quan kiểu dữ liệu & thống kê mô tả ────────────────────────────────
print("── Kiểu dữ liệu ──")
print(df_raw.dtypes.to_string())
print()

# Thống kê mô tả toàn bộ
df_raw.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T


## 3. Phân tích biến mục tiêu (target)

Biến `target = 1` nghĩa là khách hàng **bị quá hạn trên 90 ngày** trong vòng 2 năm tới — đây là định nghĩa "default" của dataset này.


In [ ]:
# ── Phân phối target ────────────────────────────────────────────────────────
target_counts = df_raw["target"].value_counts()
target_pct    = df_raw["target"].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    "Số lượng":   target_counts,
    "Tỷ lệ (%)":  target_pct.round(2),
    "Nhãn":       {0: "Good (không default)", 1: "Bad (default)"},
})
print(target_summary.to_string())

# ── Visualize ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = axes[0].bar(
    ["Good (0)", "Bad (1)"],
    target_counts.values,
    color=[PALETTE["good"], PALETTE["bad"]],
    width=0.5,
    edgecolor="white",
)
for bar, pct in zip(bars, target_pct.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 500,
        f"{pct:.1f}%",
        ha="center", va="bottom", fontsize=12, fontweight="bold",
    )
axes[0].set_title("Phân phối Target")
axes[0].set_ylabel("Số lượng khách hàng")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Pie chart
axes[1].pie(
    target_counts.values,
    labels=["Good (93.3%)", "Bad (6.7%)"],
    colors=[PALETTE["good"], PALETTE["bad"]],
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 11},
)
axes[1].set_title("Tỷ lệ Good/Bad")

plt.suptitle("Phân tích biến mục tiêu", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("01_target_distribution.png", bbox_inches="tight")
plt.show()

print()
print("⚠️  Nhận xét: Dataset bị IMBALANCED — tỷ lệ Bad chỉ 6.7%.")
print("   → Khi train model sẽ dùng class_weight='balanced' để bù lại.")


## 4. Phân tích Missing Values

In [ ]:
# ── Bảng tổng hợp missing ───────────────────────────────────────────────────
def missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Trả về DataFrame tổng hợp missing values, sắp xếp theo tỷ lệ giảm dần.
    Bao gồm dtype và gợi ý xử lý.
    """
    total   = df.isnull().sum()
    percent = total / len(df) * 100
    dtype   = df.dtypes

    summary = pd.DataFrame({
        "Missing count": total,
        "Missing %":     percent.round(2),
        "Dtype":         dtype,
    }).sort_values("Missing %", ascending=False)

    # Gợi ý xử lý tự động
    def suggest(row):
        if row["Missing %"] == 0:
            return "—"
        if row["Missing %"] > 30:
            return "Cân nhắc loại bỏ hoặc flag riêng"
        if row["Dtype"] in ["float64", "int64"]:
            return "Điền median (robust với outliers)"
        return "Điền mode"

    summary["Gợi ý xử lý"] = summary.apply(suggest, axis=1)
    return summary[summary["Missing count"] > 0]

miss_df = missing_summary(df_raw)
print(f"Số cột có missing: {len(miss_df)}")
print()
print(miss_df.to_string())


In [ ]:
# ── Visualize missing values ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

miss_plot = miss_df[miss_df["Missing %"] > 0].sort_values("Missing %")
bars = ax.barh(
    miss_plot.index,
    miss_plot["Missing %"],
    color=["#E53935" if v > 10 else "#FB8C00" for v in miss_plot["Missing %"]],
    height=0.5,
    edgecolor="white",
)
for bar, val in zip(bars, miss_plot["Missing %"]):
    ax.text(val + 0.2, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%", va="center", fontsize=11, fontweight="bold")

ax.axvline(5, color="gray", linestyle="--", linewidth=1, alpha=0.6, label="5% threshold")
ax.set_xlabel("Tỷ lệ Missing (%)")
ax.set_title("Missing Values theo từng cột")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("02_missing_values.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── Kiểm tra xem missing có liên quan đến target không ──────────────────────
# Nếu missing tập trung ở nhóm Bad → đây là thông tin có giá trị (MAR/MNAR)

for col in ["monthly_income", "dependents"]:
    miss_flag = df_raw[col].isnull()
    bad_rate_missing  = df_raw.loc[miss_flag,  "target"].mean() * 100
    bad_rate_present  = df_raw.loc[~miss_flag, "target"].mean() * 100
    print(f"[{col}]")
    print(f"  Bad rate khi MISSING  : {bad_rate_missing:.2f}%")
    print(f"  Bad rate khi CÓ GIÁ TRỊ: {bad_rate_present:.2f}%")
    diff = bad_rate_missing - bad_rate_present
    if abs(diff) > 1:
        print(f"  ⚠️  Chênh lệch {diff:+.2f}% → Missing có tương quan với target (MAR/MNAR)")
        print(f"     → Nên tạo thêm cột flag: {col}_is_missing")
    else:
        print(f"  ✓  Chênh lệch nhỏ → Missing ngẫu nhiên (MCAR), điền median là đủ")
    print()


## 5. Phân tích Outliers

Dùng hai phương pháp: **IQR** (phát hiện outlier dựa trên tứ phân vị) và **percentile** để hiểu mức độ nghiêm trọng trước khi xử lý.


In [ ]:
# ── Hàm phân tích outlier ────────────────────────────────────────────────────
def outlier_report(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """
    Báo cáo outlier theo phương pháp IQR cho danh sách cột số.
    Trả về: số lượng, tỷ lệ outlier, giá trị min/max, và ngưỡng fence.
    """
    rows = []
    for col in cols:
        s   = df[col].dropna()
        q1  = s.quantile(0.25)
        q3  = s.quantile(0.75)
        iqr = q3 - q1
        lo  = q1 - 1.5 * iqr
        hi  = q3 + 1.5 * iqr
        n_out = ((s < lo) | (s > hi)).sum()
        rows.append({
            "Cột":           col,
            "Min thực tế":   round(s.min(), 2),
            "Max thực tế":   round(s.max(), 2),
            "Fence dưới":    round(lo, 2),
            "Fence trên":    round(hi, 2),
            "Số outlier":    n_out,
            "Tỷ lệ (%)":     round(n_out / len(s) * 100, 2),
        })
    return pd.DataFrame(rows).set_index("Cột").sort_values("Tỷ lệ (%)", ascending=False)

NUM_COLS = [c for c in df_raw.columns if c != "target"]
outlier_df = outlier_report(df_raw, NUM_COLS)
print(outlier_df.to_string())


In [ ]:
# ── Boxplot toàn bộ features ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(NUM_COLS):
    good = df_raw.loc[df_raw["target"] == 0, col].dropna()
    bad  = df_raw.loc[df_raw["target"] == 1, col].dropna()
    axes[i].boxplot(
        [good, bad],
        labels=["Good", "Bad"],
        patch_artist=True,
        boxprops=dict(facecolor="none"),
        medianprops=dict(color="black", linewidth=2),
        flierprops=dict(marker=".", markersize=2, alpha=0.3),
        widths=0.5,
    )
    # Tô màu
    axes[i].findobj(plt.Polygon)[0].set_facecolor(PALETTE["good"] + "33")
    axes[i].findobj(plt.Polygon)[1].set_facecolor(PALETTE["bad"]  + "33")
    axes[i].set_title(col, fontsize=10)
    axes[i].tick_params(labelsize=9)

# Tắt axis thừa nếu có
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Boxplot: Good vs Bad theo từng feature", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("03_boxplot_features.png", bbox_inches="tight")
plt.show()


## 6. Xử lý Missing Values

Chiến lược:
- `monthly_income`: tạo cột flag `monthly_income_missing` trước, rồi điền median theo nhóm target
- `dependents`: điền 0 (không có người phụ thuộc là giá trị hợp lý nhất khi missing)


In [ ]:
# ── Làm việc trên bản copy để giữ nguyên raw data ───────────────────────────
df = df_raw.copy()

# ── 6.1: Tạo flag cho monthly_income trước khi điền ─────────────────────────
df["monthly_income_missing"] = df["monthly_income"].isnull().astype(int)
print(f"Flag 'monthly_income_missing' created: {df['monthly_income_missing'].sum():,} records flagged")

# ── 6.2: Điền monthly_income bằng median (không dùng mean vì bị ảnh hưởng bởi outlier) ──
median_income = df["monthly_income"].median()
df["monthly_income"] = df["monthly_income"].fillna(median_income)
print(f"monthly_income: điền median = {median_income:,.0f}")

# ── 6.3: Điền dependents = 0 ────────────────────────────────────────────────
df["dependents"] = df["dependents"].fillna(0).astype(int)
print(f"dependents: điền 0 cho {df_raw['dependents'].isnull().sum():,} records")

# ── Kiểm tra: không còn missing ─────────────────────────────────────────────
remaining_missing = df.isnull().sum().sum()
print(f"\n✓ Tổng missing còn lại: {remaining_missing}")
assert remaining_missing == 0, "Vẫn còn missing values — kiểm tra lại!"


## 7. Xử lý Outliers — Winsorization

**Winsorization** là phương pháp cap giá trị tại ngưỡng percentile thay vì xóa hàng.  
Ưu điểm: giữ nguyên số lượng records, không bóp méo phân phối, chuẩn trong credit scoring.

| Cột có outlier nghiêm trọng | Ngưỡng xử lý |
|---|---|
| `revolving_util` | cap tại 99th percentile (giá trị > 1 là lỗi data) |
| `debt_ratio` | cap tại 99th percentile |
| `monthly_income` | cap tại 1st–99th percentile |
| `past_due_*`, `late_90_days` | cap tại 99th percentile |


In [ ]:
# ── Hàm winsorize có logging ─────────────────────────────────────────────────
def winsorize_col(df: pd.DataFrame, col: str,
                  lower: float = 0.0, upper: float = 0.01) -> pd.DataFrame:
    """
    Cap giá trị của cột `col` tại ngưỡng percentile [lower, 1-upper].
    
    Parameters
    ----------
    lower : float — phần trăm bị cap từ phía dưới (0.01 = 1st pct)
    upper : float — phần trăm bị cap từ phía trên (0.01 = 99th pct)
    """
    lo_val = df[col].quantile(lower)    if lower > 0 else -np.inf
    hi_val = df[col].quantile(1-upper)  if upper > 0 else  np.inf

    n_affected = ((df[col] < lo_val) | (df[col] > hi_val)).sum()
    df[col] = df[col].clip(lower=lo_val, upper=hi_val)

    print(f"  {col:35s}  cap [{lo_val:>10.2f}, {hi_val:>12.2f}]  → {n_affected:,} records capped")
    return df


print("Winsorization log:")
print("─" * 80)

# Cột chỉ cap trên (lower = 0)
UPPER_ONLY = [
    "revolving_util", "debt_ratio",
    "past_due_30_59", "past_due_60_89", "late_90_days",
    "open_credit_lines", "real_estate_loans", "dependents",
]
for col in UPPER_ONLY:
    df = winsorize_col(df, col, lower=0.0, upper=0.01)

# monthly_income: cap cả hai phía
df = winsorize_col(df, "monthly_income", lower=0.01, upper=0.01)

# age: giá trị hợp lý là 18–95, xử lý cứng
n_age = ((df["age"] < 18) | (df["age"] > 95)).sum()
df["age"] = df["age"].clip(18, 95)
print(f"  {'age':35s}  clip [18, 95]  → {n_age:,} records capped")

print("─" * 80)
print("✓ Winsorization hoàn tất")


In [ ]:
# ── So sánh phân phối trước / sau winsorization ──────────────────────────────
COMPARE_COLS = ["revolving_util", "debt_ratio", "monthly_income", "past_due_30_59"]

fig, axes = plt.subplots(2, len(COMPARE_COLS), figsize=(18, 8))

for i, col in enumerate(COMPARE_COLS):
    # Trước winsorize (từ df_raw, sau khi đã điền missing)
    before = df_raw[col].dropna().clip(
        df_raw[col].quantile(0.001), df_raw[col].quantile(0.999)
    )
    after = df[col]

    # Histogram trước
    axes[0, i].hist(before, bins=60, color=PALETTE["neutral"], edgecolor="none", alpha=0.8)
    axes[0, i].set_title(f"{col}\nTrước xử lý", fontsize=10)

    # Histogram sau
    axes[1, i].hist(after, bins=60, color=PALETTE["good"], edgecolor="none", alpha=0.8)
    axes[1, i].set_title(f"Sau Winsorization", fontsize=10)

plt.suptitle("Phân phối trước và sau Winsorization (4 cột đại diện)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("04_winsorization_comparison.png", bbox_inches="tight")
plt.show()


## 8. Phân tích phân phối feature theo Target

Bước này giúp hiểu trực quan feature nào có **khả năng phân tách Good/Bad** cao — cơ sở để kỳ vọng IV cao ở bước WOE.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 9))
axes = axes.flatten()

for i, col in enumerate(NUM_COLS):
    good_vals = df.loc[df["target"] == 0, col]
    bad_vals  = df.loc[df["target"] == 1, col]

    axes[i].hist(good_vals, bins=50, alpha=0.6, color=PALETTE["good"],
                 label="Good", density=True, edgecolor="none")
    axes[i].hist(bad_vals,  bins=50, alpha=0.6, color=PALETTE["bad"],
                 label="Bad",  density=True, edgecolor="none")
    axes[i].set_title(col, fontsize=10)
    axes[i].legend(fontsize=8, frameon=False)
    axes[i].tick_params(labelsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Phân phối Good vs Bad theo từng feature (sau làm sạch)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("05_feature_distribution_by_target.png", bbox_inches="tight")
plt.show()

print("Nhận xét: Các cột 'past_due_*' và 'late_90_days' có sự tách biệt rõ nhất")
print("→ Kỳ vọng IV cao ở bước WOE tiếp theo")


## 9. Correlation Matrix — Kiểm tra multicollinearity

In [ ]:
corr = df[NUM_COLS + ["target"]].corr()

# Mask tam giác trên để dễ đọc
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    linecolor="white",
    annot_kws={"size": 9},
    ax=ax,
    cbar_kws={"shrink": 0.8},
)
ax.set_title("Correlation Matrix (sau làm sạch dữ liệu)", fontsize=13, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig("06_correlation_matrix.png", bbox_inches="tight")
plt.show()

# Phát hiện cặp multicollinear (|corr| > 0.7, loại trừ cặp với target)
print("\nCặp feature có |correlation| > 0.7 (nguy cơ multicollinearity):")
threshold = 0.7
found = False
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        c1, c2 = corr.columns[i], corr.columns[j]
        if c1 == "target" or c2 == "target":
            continue
        val = corr.iloc[i, j]
        if abs(val) > threshold:
            print(f"  {c1:35s} ↔ {c2:35s}  r = {val:.3f}")
            found = True
if not found:
    print("  ✓ Không phát hiện multicollinearity nghiêm trọng")


### Lưu dữ liệu sạch & tổng kết

In [ ]:
# ── Kiểm tra lần cuối trước khi lưu ─────────────────────────────────────────
print("=== KIỂM TRA CHẤT LƯỢNG DỮ LIỆU SAU XỬ LÝ ===")
print(f"  Shape          : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Missing values : {df.isnull().sum().sum()}")
print(f"  Duplicate rows : {df.duplicated().sum():,}")
print(f"  Target (0/1)   : {(df['target']==0).sum():,} / {(df['target']==1).sum():,}")
print(f"  Bad rate       : {df['target'].mean()*100:.2f}%")
print()
print("── Thống kê mô tả sau làm sạch ──")
print(df.describe(percentiles=[0.01, 0.5, 0.99]).T.round(3).to_string())


In [ ]:
# ── Lưu file ─────────────────────────────────────────────────────────────────
OUTPUT_PATH = "cs_training_cleaned.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"✓ Đã lưu: {OUTPUT_PATH}")
print()
print("=== TÓM TẮT BƯỚC 1 ===")
print(f"  Raw data  : {df_raw.shape[0]:,} rows, {df_raw.isnull().sum().sum():,} missing values")
print(f"  Clean data: {df.shape[0]:,} rows, {df.isnull().sum().sum()} missing values")
print(f"  Thêm mới  : cột 'monthly_income_missing' (flag MAR)")
print(f"  Winsorized: {len(UPPER_ONLY) + 2} cột")
print()
print("─── Bước tiếp theo: Bước 2 — WOE / IV Encoding ───")
